# Best-response evaders on pursuit-evasion
For every frozen pursuer, a fresh recurrent-PPO evader is trained from scratch against it under one shared budget, following the approximate-exploitability protocol of Timbers et al. (2022). These evaders are the fourth member of the evaluation suite in `eval.ipynb`.

- **Evader interface**, identical for every pursuer: a GRU over [true state, previous move one-hot], conditioned on the full $\theta$ (layout and exit).
- **Pursuers**: `Critic P1` (the Oracle, the critic's equilibrium row with privileged access to the state and context), `Trained P1` (our method), `RPPO`, `DR_RPPO`, `VARIBAD`, `PEARL`, `MMD`, and `Random P1`. Each pursuer samples its move from its own distribution, as at deployment.
- **Output**: `compared/BR_attacker_<pursuer>.pth`, holding the weights, the configuration, and the pursuer's return per update.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
BR_TOTAL_EPISODES = 1000     # same budget for every pursuer
EPISODES_PER_UPD  = 10
GAMMA             = 0.99
GAE_LAMBDA        = 0.95
PPO_EPOCHS        = 4
CLIP_EPS          = 0.2
ENT_COEF          = 0.01
VF_COEF           = 0.5
LR                = 3e-4
CKPT_EVERY        = 200
SEED_BASE         = 777
SAVE_DIR          = "compared"
os.makedirs(SAVE_DIR, exist_ok=True)

## Frozen pursuers (the wrappers are defined in `Initialize.ipynb`)

In [ ]:
Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(DefenderActor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)

print("Loaded baselines:", list(BASELINES.keys()))
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']

## Rollouts and PPO (`BRAttacker` is defined in `Initialize.ipynb`; episodes are padded to 40 steps and masked)

In [ ]:
def gae(rews, vals):
    T = len(rews)
    adv = np.zeros(T, dtype=np.float32); last = 0.0
    for t in reversed(range(T)):
        nv = vals[t + 1] if t + 1 < T else 0.0           # capture and the horizon are both terminal
        delta = rews[t] + GAMMA * nv - vals[t]
        last = delta + GAMMA * GAE_LAMBDA * last
        adv[t] = last
    return adv, adv + np.asarray(vals, dtype=np.float32)


def rollout_br(attacker, defw):
    Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col = sample_config()
    defw.reset(Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col)
    theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
    h = None
    u_prev = np.zeros(n_u, dtype=np.float32)
    o_l, u_l, lp_l, v_l, ar_l, dr_l = [], [], [], [], [], []
    for t in range(max_steps):
        x = build_x(p_row, p_col, e_row, e_col)
        a_dist = defw.act_dist(x, p_row, p_col)
        a = int(np.random.choice(n_a, p=a_dist))            # the pursuer samples from its own policy
        obs = np.concatenate([x, u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, val, h = attacker(torch.tensor(obs, device=device).view(1, 1, 2 * d + n_u), theta_t, h)
        dist = Categorical(logits=logits.view(-1))
        u = int(dist.sample())
        lp = float(dist.log_prob(torch.tensor(u, device=device)))
        new_p_row, new_p_col, new_e_row, new_e_col = next_state(p_row, p_col, e_row, e_col, a, u, Epi_Layout)
        Captured = is_capture((p_row, p_col), (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col))
        timeout = (t == max_steps - 1)
        done = Captured or timeout
        r = float(Reward(new_p_row, new_p_col, new_e_row, new_e_col, exit_rc, timeout, Captured))
        y = Observation(new_e_row, new_e_col)
        defw.observe(a, u, (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col), y)
        o_l.append(obs); u_l.append(u); lp_l.append(lp); v_l.append(val.item())
        ar_l.append(-r); dr_l.append(r)                      # the evader maximizes -r
        p_row, p_col, e_row, e_col = new_p_row, new_p_col, new_e_row, new_e_col
        u_prev = np.zeros(n_u, dtype=np.float32); u_prev[u] = 1.0
        if done:
            break
    adv, ret = gae(np.array(ar_l, dtype=np.float32), np.array(v_l, dtype=np.float32))
    return {'obs': np.stack(o_l), 'acts': np.array(u_l), 'logp': np.array(lp_l, dtype=np.float32),
            'adv': adv, 'ret': ret, 'theta': Theta.astype(np.float32),
            'def_rews': np.array(dr_l, dtype=np.float32), 'len': len(u_l)}


def pad_batch_br(eps, keys):
    E = len(eps)
    out = []
    for k in keys:
        dt = torch.long if k == 'acts' else torch.float32
        sample = np.asarray(eps[0][k])
        arr = np.zeros((E, max_steps) + sample.shape[1:], dtype=np.int64 if k == 'acts' else np.float32)
        for i, e in enumerate(eps):
            arr[i, :e['len']] = e[k]
        out.append(torch.tensor(arr, dtype=dt, device=device))
    return out


def ppo_update_br(att, opt, eps):
    obs, acts, oldlp, adv, ret = pad_batch_br(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
    theta = torch.tensor(np.stack([e['theta'] for e in eps]), dtype=torch.float32,
                         device=device).unsqueeze(1).expand(-1, max_steps, -1)
    mask = torch.zeros(len(eps), max_steps, dtype=torch.float32, device=device)
    for i, e in enumerate(eps):
        mask[i, :e['len']] = 1.0
    mv = mask > 0
    adv = (adv - adv[mv].mean()) / (adv[mv].std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values, _ = att(obs, theta)
        dist = Categorical(logits=logits)
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg_t = -torch.min(ratio * adv, torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv)
        pg = (pg_t * mask).sum() / mask.sum()
        vl = (((values - ret) ** 2) * mask).sum() / mask.sum()
        ent = (dist.entropy() * mask).sum() / mask.sum()
        loss = pg + VF_COEF * vl - ENT_COEF * ent
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(att.parameters(), 1.0)
        opt.step()


def train_br(def_name, seed):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    defw = get_defender(def_name)
    att = BRAttacker().to(device)
    opt = optim.AdamW(att.parameters(), lr=LR)
    n_upd = BR_TOTAL_EPISODES // EPISODES_PER_UPD
    def_hist = []
    fname = 'BR_attacker_' + def_name.replace(' ', '_') + '.pth'
    for upd in tqdm(range(n_upd), desc='BR vs ' + def_name):
        eps = [rollout_br(att, defw) for _ in range(EPISODES_PER_UPD)]
        ppo_update_br(att, opt, eps)
        def_hist.append(float(np.mean([e['def_rews'].sum() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'attacker': att.state_dict(), 'defender': def_name,
                        'config': {'RNN_H': RNN_H, 'seed': seed, 'budget_episodes': BR_TOTAL_EPISODES},
                        'defender_return_per_episode': def_hist},
                       os.path.join(SAVE_DIR, fname))
    return att, def_hist

## Train one evader per pursuer

In [ ]:
BR_ATTACKERS, BR_CURVES = {}, {}
for k, name in enumerate(DEFENDERS):
    BR_ATTACKERS[name], BR_CURVES[name] = train_br(name, seed=SEED_BASE + k)

## Convergence check: each pursuer's return should plateau, otherwise increase `BR_TOTAL_EPISODES`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, c in BR_CURVES.items():
    ax.plot(np.arange(1, len(c) + 1), c, label=name)
ax.set_xlabel('PPO update'); ax.set_ylabel("pursuer's return per episode")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig('br_curves.jpg', dpi=300, bbox_inches='tight'); plt.show()